# Part 2 — Minimum Spanning Tree: Kruskal's & Prim's
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---

### How to use this notebook
Run each cell top-to-bottom with **Shift+Enter**.

| Cell | What it does |
|------|-------------|
| **Cell 1** | Union-Find, adjacency list, Kruskal's & Prim's definitions |
| **Cell 2** | Full step-by-step simulation with exact spec output format |
| **Cell 3** | NetworkX graph visualization (original + MSTs side-by-side) |
| **Cell 4** | Automated test suite — 5 named graphs, cost agreement check |

---

### Algorithm Explanations

#### Kruskal's Algorithm
- **Strategy**: Greedy edge selection. Sort all edges by weight (ascending), then add the cheapest edge that does **not** create a cycle.
- **Cycle detection**: Union-Find (Disjoint Set Union) — `find(u)==find(v)` means they're already connected → skip.
- **Data structures**: `Edge list` (sorted) + `Union-Find` with path compression + union by rank.
- **Time**: O(E log E) — dominated by sorting.

#### Prim's Algorithm
- **Strategy**: Greedy vertex expansion. Start from one vertex, always pick the cheapest edge connecting the visited set to an unvisited vertex.
- **Data structures**: `Adjacency list` + `Min-Heap` (Python `heapq`).
- **Time**: O(E log V) with binary heap.

---

### Flowcharts

**Kruskal's**
```
Sort all edges by weight
Initialize Union-Find (each vertex is its own component)
for each edge (u, v, w) in sorted order:
    if find(u) != find(v):           ← no cycle
        union(u, v)
        add (u, v, w) to MST
    else:
        SKIP                         ← would form a cycle
    if |MST| == V-1: STOP
```

**Prim's**
```
visited = { start_vertex }
push all edges from start into min-heap
while heap not empty:
    (w, u, v) = pop minimum
    if v already visited: skip
    mark v visited
    add (u, v, w) to MST
    push all unvisited neighbors of v into heap
```

---

### Union-Find Explained
```
parent = [0, 1, 2, 3, 4]   # initially each node is its own root

find(3):
  parent[3] = 3 → root is 3

union(1, 2):
  parent[2] = 1   → component {1, 2}

union(1, 3):
  parent[3] = 1   → component {1, 2, 3}

find(3) with path compression:
  parent[3] → parent[1] → 1  (root)
  path compression: parent[3] = 1  (direct link)
```

In [1]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 1 — DATA STRUCTURES & ALGORITHM DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════
import heapq
import random
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "monospace",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})

random.seed(42)


# ════════════════════════════════════════════════════════════════════════
#  DATA STRUCTURE 1 — UNION-FIND (Disjoint Set Union)
#  Used by Kruskal's for O(α(n)) ≈ O(1) cycle detection.
#
#  Two optimisations:
#    1. Path compression  — find() flattens the tree toward root
#    2. Union by rank     — attach shorter tree under taller tree
# ════════════════════════════════════════════════════════════════════════

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))   # each node starts as its own root
        self.rank   = [0] * n          # rank keeps tree shallow

    def find(self, x):
        """Return root of x. Path compression applied recursively."""
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # compress path
        return self.parent[x]

    def union(self, x, y):
        """Merge components. Returns False if already in same set (= cycle)."""
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                               # same component → cycle!
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx                            # make rx the taller tree
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True


# ════════════════════════════════════════════════════════════════════════
#  DATA STRUCTURE 2 — ADJACENCY LIST
#  dict { vertex: [(neighbor, weight), ...] }
#  Space: O(V + E)   |   Neighbor lookup: O(degree(v))
# ════════════════════════════════════════════════════════════════════════

def build_adjacency_list(vertices, edges):
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))     # undirected
    return dict(adj)


def print_graph_structure(vertices, edges):
    """Display the graph's adjacency list — spec requirement."""
    adj = build_adjacency_list(vertices, edges)
    print("  Graph Structure — Adjacency List:")
    print(f"  Vertices : {vertices}")
    print(f"  Edges    : {len(edges)}")
    print()
    for v in sorted(vertices):
        nbrs = ", ".join(f"{nb}(w={wt})" for nb, wt in sorted(adj.get(v,[])))
        print(f"    {v:>3} → [{nbrs}]")
    print()
    print("  Edge list (sorted by weight ascending):")
    for u, v, w in sorted(edges, key=lambda e: e[2]):
        print(f"    ({u}–{v}, w={w})")
    print()


# ════════════════════════════════════════════════════════════════════════
#  KRUSKAL'S ALGORITHM
#  Data structures: sorted Edge List + Union-Find
#  Time: O(E log E)
# ════════════════════════════════════════════════════════════════════════

def kruskal(vertices, edges, verbose=True):
    """
    Build MST using Kruskal's algorithm.
    Returns (mst_edges, total_cost).
    """
    n       = len(vertices)
    v_idx   = {v: i for i, v in enumerate(vertices)}
    uf      = UnionFind(n)
    mst     = []
    sorted_edges = sorted(edges, key=lambda e: e[2])    # ← edge sorting step

    if verbose:
        print("═" * 62)
        print("  KRUSKAL'S ALGORITHM")
        print("═" * 62)
        print_graph_structure(vertices, edges)
        print("  Edge Selection + Cycle Detection:")
        print("─" * 62)

    step = 0
    for u, v, w in sorted_edges:
        step += 1
        added = uf.union(v_idx[u], v_idx[v])

        if added:
            mst.append((u, v, w))
            if verbose:
                mst_view = [(a,b,c) for a,b,c in mst]
                print(f"  Step {step:>2} | ({u}–{v}, w={w:>2}) → ✅ ADDED   "
                      f"MST so far: {mst_view}")
        else:
            if verbose:
                print(f"  Step {step:>2} | ({u}–{v}, w={w:>2}) → ❌ SKIPPED "
                      f"(find({u})==find({v}) → cycle)")

        if len(mst) == n - 1:
            break

    cost = sum(w for _, _, w in mst)

    if verbose:
        print()
        print("  Edges selected for MST:")
        for u, v, w in mst:
            print(f"    {u} -- {v}  (weight {w})")
        print(f"  Total MST Cost = {cost}")
        print("═" * 62)

    return mst, cost


# ════════════════════════════════════════════════════════════════════════
#  PRIM'S ALGORITHM
#  Data structures: Adjacency List + Min-Heap (heapq)
#  Time: O(E log V)
# ════════════════════════════════════════════════════════════════════════

def prim(vertices, edges, start=None, verbose=True):
    """
    Build MST using Prim's algorithm.
    Returns (mst_edges, total_cost).
    """
    adj   = build_adjacency_list(vertices, edges)
    start = start if start is not None else vertices[0]

    visited = set()
    mst     = []
    heap    = [(0, None, start)]   # (weight, from_vertex, to_vertex)

    if verbose:
        print("═" * 62)
        print("  PRIM'S ALGORITHM")
        print("═" * 62)
        print_graph_structure(vertices, edges)
        print(f"  Starting vertex : {start}")
        print("─" * 62)
        print("  Growing MST step by step:")

    step = 0
    while heap:
        w, u, v = heapq.heappop(heap)   # extract minimum-weight border edge
        if v in visited:
            continue

        visited.add(v)

        if u is not None:               # skip the dummy start entry
            mst.append((u, v, w))
            step += 1
            if verbose:
                print(f"  Step {step:>2} | Add ({u}–{v}, w={w:>2})"
                      f"   Visited: {sorted(visited)}")

        for neighbor, weight in sorted(adj.get(v, [])):
            if neighbor not in visited:
                heapq.heappush(heap, (weight, v, neighbor))

    cost = sum(c for _, _, c in mst)

    if verbose:
        print()
        print("  Edges selected for MST:")
        for u, v, w in mst:
            print(f"    {u} -- {v}  (weight {w})")
        print(f"  Total MST Cost = {cost}")
        print("═" * 62)

    return mst, cost


print("✅  MST algorithms loaded")
print("    Data structures: Union-Find | Adjacency List | Min-Heap (heapq)")


# ── VISUALIZATION STYLE ────────────────────────────────────────────────
# Seaborn + matplotlib — consistent professional style across all charts.
import seaborn as sns

PALETTE   = sns.color_palette("deep", 10)
PAL_NAMES = {n: PALETTE[i] for i, n in enumerate([
    "Bubble Sort","Selection Sort","Insertion Sort","Merge Sort",
    "Quick Sort","Random-Quick Sort","Counting Sort","Radix Sort",
    "Kruskal","Prim"
])}
GREEN  = "#10B981"   # fastest / best
BLUE   = "#2563EB"   # primary accent
SLATE  = "#64748B"   # muted labels
FIG_BG = "#F8FAFC"   # figure background

sns.set_theme(style="whitegrid", font_scale=1.05)
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor":  FIG_BG,
    "axes.facecolor":    "white",
    "axes.edgecolor":    "#E2E8F0",
    "axes.linewidth":    0.8,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlepad":     12,
    "axes.labelsize":    10,
    "axes.labelcolor":   "#374151",
    "axes.labelpad":     6,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "xtick.color":       "#6B7280",
    "ytick.color":       "#6B7280",
    "grid.color":        "#E5E7EB",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.8,
    "legend.frameon":    True,
    "legend.framealpha": 0.92,
    "legend.fontsize":   9,
    "legend.edgecolor":  "#E2E8F0",
    "figure.dpi":        130,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
    "font.family":       "sans-serif",
    "font.size":         10,
})

def style_ax(ax, remove_spines=("top","right")):
    """Apply consistent spine cleanup to any axes."""
    for sp in remove_spines:
        ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

def annotate_bars_h(ax, bars, fmt=".2f", offset_frac=0.015):
    """Add value labels to horizontal bars."""
    xmax = ax.get_xlim()[1]
    for bar in bars:
        w = bar.get_width()
        label = f"{w:{fmt}}" if 'f' in fmt else f"{int(w):,}"
        ax.text(w + xmax * offset_frac,
                bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left",
                fontsize=8.5, color="#374151", fontweight="600")

def annotate_bars_v(ax, bars, fmt=",d", offset=0.01):
    """Add value labels to vertical bars."""
    ymax = ax.get_ylim()[1]
    for bar in bars:
        h = bar.get_height()
        if h == 0: continue
        label = f"{int(h):,}" if fmt == ",d" else f"{h:{fmt}}"
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + ymax * offset,
                label, ha="center", va="bottom",
                fontsize=7.5, color="#374151", fontweight="600")


✅  MST algorithms loaded
    Data structures: Union-Find | Adjacency List | Min-Heap (heapq)


In [2]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — STEP-BY-STEP SIMULATION
#  Runs both algorithms on the same graph with full verbose output.
#  Output format matches the project spec exactly.
#
#  To use your own graph:
#    MY_VERTICES = [1, 2, 3, 4]
#    MY_EDGES    = [(1,2,3), (2,3,5), (1,3,10), (3,4,4)]
#    Then call kruskal(MY_VERTICES, MY_EDGES) and prim(MY_VERTICES, MY_EDGES)
# ═══════════════════════════════════════════════════════════════════════

VERTICES = [1, 2, 3, 4, 5, 6]
EDGES = [
    (1, 2, 4),
    (1, 3, 3),
    (2, 3, 5),
    (2, 4, 6),
    (3, 4, 7),
    (3, 5, 8),
    (4, 5, 9),
    (4, 6, 5),
    (5, 6, 6),
]

# ── Run Kruskal's ────────────────────────────────────────────────────
k_mst, k_cost = kruskal(VERTICES, EDGES, verbose=True)

print()

# ── Run Prim's from vertex 1 ─────────────────────────────────────────
p_mst, p_cost = prim(VERTICES, EDGES, start=1, verbose=True)

print()
print("─" * 40)
print(f"  Kruskal's total cost : {k_cost}")
print(f"  Prim's total cost    : {p_cost}")
agree = "✅ AGREE" if k_cost == p_cost else "❌ MISMATCH"
print(f"  Both algorithms      : {agree}")
print("─" * 40)


══════════════════════════════════════════════════════════════
  KRUSKAL'S ALGORITHM
══════════════════════════════════════════════════════════════
  Graph Structure — Adjacency List:
  Vertices : [1, 2, 3, 4, 5, 6]
  Edges    : 9

      1 → [2(w=4), 3(w=3)]
      2 → [1(w=4), 3(w=5), 4(w=6)]
      3 → [1(w=3), 2(w=5), 4(w=7), 5(w=8)]
      4 → [2(w=6), 3(w=7), 5(w=9), 6(w=5)]
      5 → [3(w=8), 4(w=9), 6(w=6)]
      6 → [4(w=5), 5(w=6)]

  Edge list (sorted by weight ascending):
    (1–3, w=3)
    (1–2, w=4)
    (2–3, w=5)
    (4–6, w=5)
    (2–4, w=6)
    (5–6, w=6)
    (3–4, w=7)
    (3–5, w=8)
    (4–5, w=9)

  Edge Selection + Cycle Detection:
──────────────────────────────────────────────────────────────
  Step  1 | (1–3, w= 3) → ✅ ADDED   MST so far: [(1, 3, 3)]
  Step  2 | (1–2, w= 4) → ✅ ADDED   MST so far: [(1, 3, 3), (1, 2, 4)]
  Step  3 | (2–3, w= 5) → ❌ SKIPPED (find(2)==find(3) → cycle)
  Step  4 | (4–6, w= 5) → ✅ ADDED   MST so far: [(1, 3, 3), (1, 2, 4), (4, 6, 5)]
  St

In [3]:
# =========================================================
#  CELL 3 - GRAPH VISUALIZATION
#  Left  : Full graph  |  Middle: Kruskal  |  Right: Prim
# =========================================================

def draw_graph(ax, title, vertices, edges, mst_edges=None, pos=None):
    G = nx.Graph()
    G.add_nodes_from(vertices)
    for u, v, w in edges:
        G.add_edge(u, v, weight=w)
    if pos is None:
        pos = nx.spring_layout(G, seed=42)
    mst_set = set()
    if mst_edges:
        for u, v, _ in mst_edges:
            mst_set.add((u, v)); mst_set.add((v, u))
    ecols = ["#2A9D8F" if (u,v) in mst_set else "#CBD5E1"
             for u, v in G.edges()]
    ewids = [3.5 if (u,v) in mst_set else 1.0
             for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color="#264653", node_size=700, alpha=0.95)
    nx.draw_networkx_labels(G, pos, ax=ax,
                            font_color="white", font_size=12,
                            font_family="monospace")
    nx.draw_networkx_edges(G, pos, ax=ax,
                           edge_color=ecols, width=ewids, alpha=0.9)
    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=nx.get_edge_attributes(G, "weight"),
        ax=ax, font_size=8, font_family="monospace"
    )
    ax.set_title(title, pad=10)
    ax.axis("off")
    return pos

G_ref = nx.Graph()
G_ref.add_nodes_from(VERTICES)
for u, v, w in EDGES:
    G_ref.add_edge(u, v, weight=w)
POS = nx.spring_layout(G_ref, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Minimum Spanning Tree - Kruskal vs Prim",
             fontsize=13, fontweight="bold", y=1.02)

draw_graph(axes[0], "Full Graph (all edges)",
           VERTICES, EDGES, pos=POS)
draw_graph(axes[1], f"Kruskal MST  (cost={k_cost})",
           VERTICES, EDGES, k_mst, pos=POS)
draw_graph(axes[2], f"Prim MST     (cost={p_cost})",
           VERTICES, EDGES, p_mst, pos=POS)

legend = [
    mpatches.Patch(color="#2A9D8F", label="MST edge"),
    mpatches.Patch(color="#CBD5E1", label="Non-MST edge"),
]
fig.legend(handles=legend, loc="lower center", ncol=2,
           frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig("/tmp/p2_graph_viz.png", bbox_inches="tight", dpi=110)
plt.show()
print("MST visualization saved.")

# Edge weight comparison
fig2, ax2 = plt.subplots(figsize=(9, 4))
k_ws = sorted([w for _,_,w in k_mst])
p_ws = sorted([w for _,_,w in p_mst])
xs   = range(max(len(k_ws), len(p_ws)))
ax2.bar([x-0.2 for x in xs[:len(k_ws)]], k_ws,
        width=0.35, color="#457B9D", label="Kruskal edges", alpha=0.9)
ax2.bar([x+0.2 for x in xs[:len(p_ws)]], p_ws,
        width=0.35, color="#2A9D8F", label="Prim edges", alpha=0.9)
ax2.set_xlabel("Edge rank (sorted by weight)")
ax2.set_ylabel("Weight")
ax2.set_title(f"MST Edge Weights Selected  (Kruskal={k_cost}  Prim={p_cost})", pad=8)
ax2.legend()
ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p2_edge_weights.png", bbox_inches="tight", dpi=110)
plt.show()


MST visualization saved.


/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30889/3104224618.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30889/3104224618.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 4 — AUTOMATED TEST SUITE
#  Each test checks: (1) Kruskal cost == Prim cost, (2) == expected cost.
# ═══════════════════════════════════════════════════════════════════════

TEST_GRAPHS = [
    ("Simple triangle",
     [1,2,3], [(1,2,1),(2,3,2),(1,3,10)], 3),
    ("Linear chain",
     [1,2,3,4], [(1,2,5),(2,3,3),(3,4,4)], 12),
    ("All equal weights",
     [1,2,3,4], [(1,2,1),(2,3,1),(3,4,1),(1,4,1),(1,3,1),(2,4,1)], 3),
    ("Spec sample graph (6 vertices)",
     [1,2,3,4,5,6],
     [(1,2,4),(1,3,3),(2,3,5),(2,4,6),(3,4,7),(3,5,8),(4,5,9),(4,6,5),(5,6,6)],
     24),
    ("Classic 5-vertex",
     [1,2,3,4,5], [(1,2,3),(2,4,5),(4,5,6),(1,3,4),(3,5,7)], 18),
]

print("=" * 70)
print("  MST TEST SUITE — Kruskal vs Prim cost agreement")
print("=" * 70)
print(f"  {'Test':<38} {'Kruskal':>8} {'Prim':>6} {'Expected':>9}  {'Pass?'}")
print("─" * 70)

total = passed = 0
for name, verts, edges, expected in TEST_GRAPHS:
    total += 1
    try:
        _, kc = kruskal(verts, edges, verbose=False)
        _, pc = prim(verts, edges, verbose=False)
        ok = (kc == pc == expected)
        if ok: passed += 1
        status = "✅" if ok else "❌"
        print(f"  {name:<38} {kc:>8} {pc:>6} {expected:>9}  {status}")
    except Exception as e:
        print(f"  {name:<38} ERROR: {e}")

print("─" * 70)
print(f"  Results: {passed}/{total} tests passed")
if passed == total:
    print("  🎉 Both algorithms verified correct on all graphs!")
print("=" * 70)


  MST TEST SUITE — Kruskal vs Prim cost agreement
  Test                                    Kruskal   Prim  Expected  Pass?
──────────────────────────────────────────────────────────────────────
  Simple triangle                               3      3         3  ✅
  Linear chain                                 12     12        12  ✅
  All equal weights                             3      3         3  ✅
  Spec sample graph (6 vertices)               24     24        24  ✅
  Classic 5-vertex                             18     18        18  ✅
──────────────────────────────────────────────────────────────────────
  Results: 5/5 tests passed
  🎉 Both algorithms verified correct on all graphs!
